# Leetcode 177
## Nth Higest Salary

In [0]:
# https://leetcode.com/problems/find-total-time-spent-by-each-employee
# Completed at 2026/04/22

In [0]:
# Frameworks
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from datetime import datetime

## Part 1: Parsing the raw data into PySpark Dataframes

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from datetime import datetime


def parse_raw_to_df(table_str):

    spark = SparkSession.builder.getOrCreate()

    # Step 1 — clean lines
    lines = []

    for line in table_str.splitlines():

        line = line.strip()

        if not line:
            continue

        if line.startswith("+"):
            continue

        lines.append(line)

    # Step 2 — extract schema
    schema_cols = []
    schema_types = []

    i = 0

    # find schema header
    while i < len(lines):

        if "Column Name" in lines[i]:
            i += 1
            break

        i += 1

    # read schema rows ONLY until hitting non-type row
    valid_types = {
        "int",
        "integer",
        "string",
        "varchar",
        "char",
        "date"
    }

    while i < len(lines):

        parts = [p.strip() for p in lines[i].split("|") if p.strip()]

        if len(parts) != 2:
            break

        if parts[1].lower() not in valid_types:
            break

        schema_cols.append(parts[0])
        schema_types.append(parts[1])

        i += 1

    # Step 3 — Spark type mapping
    type_map = {
        "int": IntegerType(),
        "integer": IntegerType(),
        "string": StringType(),
        "varchar": StringType(),
        "char": StringType(),
        "date": DateType()
    }

    fields = []

    for col, typ in zip(schema_cols, schema_types):

        spark_type = type_map.get(
            typ.lower(),
            StringType()
        )

        fields.append(
            StructField(col, spark_type, True)
        )

    schema = StructType(fields)

    # Step 4 — find data header
    data_start = None

    for idx, line in enumerate(lines):

        parts = [p.strip() for p in line.split("|") if p.strip()]

        if parts == schema_cols:
            data_start = idx + 1
            break

    if data_start is None:
        raise ValueError(
            f"Data section not found. Expected header: {schema_cols}"
        )

    # Step 5 — parse data rows
    data_rows = []

    for line in lines[data_start:]:

        parts = [p.strip() for p in line.split("|") if p.strip()]

        if len(parts) != len(schema_cols):
            continue

        converted_row = []

        for val, typ in zip(parts, schema_types):

            typ = typ.lower()

            if typ in ["int", "integer"]:
                converted_row.append(int(val))

            elif typ == "date":
                converted_row.append(
                    datetime.strptime(val, "%Y-%m-%d").date()
                )

            else:
                converted_row.append(val)

        data_rows.append(tuple(converted_row))

    df = spark.createDataFrame(
        data_rows,
        schema
    )

    return df

In [0]:
# Initial data tables

# EMPLOYEE
r_employees = """
+-------------+------+
| Column Name | Type |
+-------------+------+
| id          | int  |
| salary      | int  |
+-------------+------+
+----+--------+
| id | salary |
+----+--------+
| 1  | 100    |
| 2  | 200    |
| 3  | 300    |
+----+--------+
"""
employees_df = parse_raw_to_df(r_employees)
employees_df.createOrReplaceTempView("employees")

# EXTRA: variables
n = 2

In [0]:
employees_df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- salary: integer (nullable = true)



In [0]:
employees_df.display(5)

id,salary
1,100
2,200
3,300


## Part 2: Querying and Solution

Write a solution to find the nth highest distinct salary from the Employee table. If there are less than n distinct salaries, return null.

The result format is in the following example.

![image_1776830645670.png](./image_1776830645670.png "image_1776830645670.png")

### SparkSQL

In [0]:
N = 2

sol_sql = spark.sql(f"""
                        WITH ranking AS (
                            SELECT 
                                salary,
                                DENSE_RANK() OVER (ORDER BY salary DESC) AS rnk
                            FROM employees
                        )
                        SELECT salary
                        FROM ranking
                        WHERE rnk = {N}
                        LIMIT 1
                    """).show()

+------+
|salary|
+------+
|   200|
+------+



### PySpark


In [0]:
# Pyspark code
n = 2
window_spec = Window.orderBy(F.col("salary").desc())

ranking_df = employees_df.withColumn(
    "rnk",
    F.dense_rank().over(window_spec)
    )

result_df = ranking_df.filter(
    F.col("rnk") == n
).select("salary")

result_df.show()


/databricks/python/lib/python3.11/site-packages/pyspark/sql/connect/expressions.py:1017: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+------+
|salary|
+------+
|   200|
+------+

